# Chess Elo Regression — Advanced Game Features

This notebook uses `chess_features.py` to extract rich, board-aware features
from each game rather than surface-level token counts.  Every feature is
derived by replaying the game move-by-move on a real chess engine, so things
like *legal move count*, *en passant*, and *castling side* are computed
correctly rather than guessed from the PGN string.

In [ ]:
# ── dependencies ─────────────────────────────────────────────────────────────
# pip install chess zstandard lightgbm pandas scikit-learn matplotlib

import io
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zstandard as zstd
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from scipy.sparse import csr_matrix, hstack

# Our feature extractor — must be in the same directory as this notebook
from chess_features import extract_features_dataframe

print('All imports OK')

In [ ]:
# ── global configuration ─────────────────────────────────────────────────────
DATA_PATH    = "lichess_db_standard_rated_2013-03.pgn.zst"
MAX_GAMES    = 20_000   # keep low for extraction speed; raise once confident
RANDOM_STATE = 42
TEST_SIZE    = 0.2

# All features produced by chess_features.py
ADVANCED_NUMERIC = [
    "total_ply_count",
    "material_balance_end",
    "checks_given_white",
    "checks_given_black",
    "first_capture_move_white",
    "first_capture_move_black",
    "pawn_captures_total",
    "piece_captures_total",
    "castle_move_white",
    "castle_move_black",
    "result_encoded",
    "consec_same_piece_white",
    "consec_same_piece_black",
    "queen_moves_before_10",
    "white_territory_depth",
    "black_territory_depth",
    "promotions",
    "en_passant_captures",
    "legal_moves_white_move5",
    "legal_moves_black_move5",
]

# Categorical features produced by chess_features.py
ADVANCED_CATEGORICAL = [
    "castle_side_white",
    "castle_side_black",
]

# Metadata features from the PGN header (same as original notebook)
METADATA = ["TimeControl", "ECO"]

MODEL_CONFIG = {
    "n_estimators":  300,
    "learning_rate": 0.08,
    "num_leaves":     63,
    "min_child_samples": 20,
    "random_state":  RANDOM_STATE,
    "verbose":       -1,
}

## 1 · Load raw games from the compressed PGN

In [ ]:
games_list = []
dctx = zstd.ZstdDecompressor()

with open(DATA_PATH, "rb") as f:
    with dctx.stream_reader(f) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")
        current_game = {}

        for line in text_stream:
            line = line.strip()

            if line.startswith("["):
                tag = line.split(" ")[0][1:]
                value = line.split('"')[1]
                if tag in ["WhiteElo", "BlackElo", "TimeControl", "ECO",
                            "Termination", "Result"]:
                    current_game[tag] = value

            elif line.startswith("1."):
                current_game["Moves"] = line
                if "WhiteElo" in current_game and "BlackElo" in current_game:
                    games_list.append(current_game)
                current_game = {}
                if len(games_list) >= MAX_GAMES:
                    break

df_raw = pd.DataFrame(games_list)
df_raw["WhiteElo"] = pd.to_numeric(df_raw["WhiteElo"], errors="coerce")
df_raw["BlackElo"] = pd.to_numeric(df_raw["BlackElo"], errors="coerce")
df_raw = df_raw.dropna(subset=["WhiteElo", "BlackElo"]).copy()
df_raw["WhiteElo"] = df_raw["WhiteElo"].astype(int)
df_raw["BlackElo"] = df_raw["BlackElo"].astype(int)
df_raw["Moves"] = df_raw["Moves"].fillna("").astype(str)

print(f"Loaded {len(df_raw):,} games")
df_raw.head(3)

## 2 · Extract advanced features

We replay every game on a real chess board.  This is slower than regex
counting but produces features that are semantically correct — legal-move
counts, actual castling sides, en passant, territory depth, etc.

In [ ]:
t0 = time.time()
df_feats = extract_features_dataframe(df_raw)
elapsed = time.time() - t0

print(f"Feature extraction: {elapsed:.1f}s  ({elapsed/len(df_raw)*1000:.1f} ms/game)")
df_feats.head(3)

In [ ]:
# Merge with metadata and Elo targets
df = df_raw[["WhiteElo", "BlackElo"] + METADATA].join(df_feats)

print(df.shape)
df.describe()

## 3 · Feature exploration

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

plot_features = [
    "total_ply_count", "material_balance_end",
    "checks_given_white", "checks_given_black",
    "pawn_captures_total", "piece_captures_total",
    "queen_moves_before_10", "legal_moves_white_move5",
    "white_territory_depth", "black_territory_depth",
    "promotions", "en_passant_captures",
]

for ax, col in zip(axes, plot_features):
    ax.hist(df[col], bins=30, edgecolor='none', alpha=0.75)
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("count")

plt.suptitle("Distribution of advanced game features", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Castling breakdown
print("White castling side:")
print(df["castle_side_white"].value_counts())
print()
print("Black castling side:")
print(df["castle_side_black"].value_counts())

In [ ]:
# Correlation of numeric features with WhiteElo
corr = df[ADVANCED_NUMERIC + ["WhiteElo"]].corr()["WhiteElo"].drop("WhiteElo").sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
corr.plot(kind="barh", ax=ax, color=["#e05252" if v < 0 else "#4e91d9" for v in corr])
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Pearson correlation with WhiteElo")
ax.set_xlabel("correlation")
plt.tight_layout()
plt.show()

## 4 · Build the feature matrix

In [ ]:
def build_matrix(frame: pd.DataFrame) -> csr_matrix:
    """
    Stack:
      - numeric advanced features
      - one-hot encoded castling sides
      - one-hot encoded metadata (TimeControl, ECO)
    """
    parts = []

    # Numeric
    parts.append(csr_matrix(frame[ADVANCED_NUMERIC].astype(float).values))

    # Categorical: castle sides
    castle_dummies = pd.get_dummies(
        frame[ADVANCED_CATEGORICAL], drop_first=True
    ).astype(float)
    parts.append(csr_matrix(castle_dummies.values))

    # Metadata
    meta_dummies = pd.get_dummies(
        frame[METADATA], drop_first=True
    ).astype(float)
    parts.append(csr_matrix(meta_dummies.values))

    return hstack(parts)


X = build_matrix(df)
print("Feature matrix shape:", X.shape)

## 5 · Train & evaluate

In [ ]:
def run_experiment(target_col: str, label: str) -> dict:
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    model = lgb.LGBMRegressor(**MODEL_CONFIG)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    print(f"{label}  →  MAE: {mae:.1f}  RMSE: {rmse:.1f}")

    return {"model": model, "X_test": X_test, "y_test": y_test,
            "preds": preds, "mae": mae, "rmse": rmse, "label": label}


white_run = run_experiment("WhiteElo", "White Elo")
black_run = run_experiment("BlackElo", "Black Elo")

In [ ]:
# Summary table
summary = pd.DataFrame([
    {"target": r["label"], "MAE": round(r["mae"], 2), "RMSE": round(r["rmse"], 2)}
    for r in [white_run, black_run]
])
summary

## 6 · Diagnostic plots

In [ ]:
def plot_predictions(run: dict, ax):
    y_true = run["y_test"].values
    y_pred = run["preds"]
    lo, hi = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.scatter(y_true, y_pred, alpha=0.3, s=10, rasterized=True)
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5)
    ax.set_xlabel("True Elo")
    ax.set_ylabel("Predicted Elo")
    ax.set_title(f"{run['label']}  (MAE {run['mae']:.1f})")
    ax.grid(alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_predictions(white_run, axes[0])
plot_predictions(black_run, axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for White Elo model
# Build a column-name list that matches the matrix construction order
castle_dummy_cols = list(
    pd.get_dummies(df[ADVANCED_CATEGORICAL], drop_first=True).columns
)
meta_dummy_cols = list(
    pd.get_dummies(df[METADATA], drop_first=True).columns
)
all_feature_names = ADVANCED_NUMERIC + castle_dummy_cols + meta_dummy_cols

model = white_run["model"]
importance = pd.Series(
    model.feature_importances_[:len(all_feature_names)],
    index=all_feature_names
).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(9, 7))
importance.plot(kind="barh", ax=ax, color="#4e91d9")
ax.set_title("Top 20 feature importances — White Elo model")
ax.set_xlabel("LightGBM importance (split count)")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for run, ax in zip([white_run, black_run], axes):
    residuals = run["y_test"].values - run["preds"]
    ax.hist(residuals, bins=60, edgecolor='none', alpha=0.75)
    ax.axvline(0, color='red', lw=1.5)
    ax.set_title(f"Residuals — {run['label']}")
    ax.set_xlabel("True − Predicted Elo")
    ax.set_ylabel("count")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7 · Notes on feature engineering choices

| Feature | Why it matters |
|---|---|
| `legal_moves_white/black_move5` | Opening complexity — stronger players typically reach positions with more choices |
| `material_balance_end` | Large imbalances correlate with decisive games and blunder-prone play |
| `white/black_territory_depth` | Piece activity proxy — aggressive players push pieces deep into enemy half |
| `queen_moves_before_10` | Early queen sorties are a beginner hallmark |
| `consec_same_piece_*` | Repeated piece shuffling may indicate hesitation or tactical blindness |
| `castle_side_*` | Encodes opening identity (king-side castle is far more common) |
| `en_passant_captures` | Rare; awareness of the rule is a mild skill indicator |
| `first_capture_move_*` | Early captures often correlate with specific opening families |

**Key implementation detail:** legal-move counts are obtained from
`board.legal_moves.count()` *before* the move is pushed.  There is no
shortcut — you must replay the game to get this right.